# CS273P Final Project — Demo Notebook
## Multi-Task Learning for Predictive Maintenance

This notebook demonstrates the full pipeline:
1. Load and inspect the dataset
2. Load the best trained checkpoint
3. Run inference on the test set
4. Visualize results (loss curves, confusion matrix)
5. Run inference on a custom input

> **Prerequisites:** Run `python src/train.py` from the project root before running this notebook.

## 0. Setup

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
os.chdir(PROJECT_ROOT)

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from dataset  import load_data, get_dataloaders
from model    import MTLModel, MTLLoss, count_parameters
from evaluate import evaluate, compute_metrics, print_report, plot_confusion_matrix
from utils    import set_seed, get_device

set_seed(42)
device = get_device()
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 1. Dataset Overview

In [ ]:
DATA_PATH = os.path.join('data', 'ai4i2020.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Machine failure'].value_counts()
axes[0].bar(['No Failure', 'Failure'], counts.values, color=['#2A9D8F', '#E76F51'])
axes[0].set_title('Binary Failure Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

type_counts = [df[col].sum() for col in failure_cols]
axes[1].bar(failure_cols, type_counts, color='#E9C46A')
axes[1].set_title('Failure Type Breakdown')
axes[1].set_ylabel('Count')
for i, v in enumerate(type_counts):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Overall failure rate: {df["Machine failure"].mean()*100:.1f}%')

## 2. Load Data + Model

In [ ]:
train_ds, val_ds, test_ds, scaler, w_bin, w_type = load_data(DATA_PATH)
train_loader, val_loader, test_loader = get_dataloaders(train_ds, val_ds, test_ds, batch_size=64)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
CHECKPOINT = os.path.join('checkpoints', 'best_model.pt')

if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(
        f'Checkpoint not found at {CHECKPOINT}.\n'
        'Please run `python src/train.py` from the project root first.'
    )

model = MTLModel().to(device)
checkpoint = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f'Loaded checkpoint from epoch {checkpoint["epoch"]}')
print(f'Trainable parameters: {count_parameters(model):,}')
print(f'Val metrics at save time: {checkpoint["metrics"]}')

## 3. Test Set Evaluation

In [ ]:
loss_fn = MTLLoss(alpha=0.5, binary_pos_weight=w_bin.to(device), type_class_weights=w_type.to(device))

test_loss, bin_preds, bin_targets, type_preds, type_targets = evaluate(
    model, test_loader, loss_fn, device
)
metrics = compute_metrics(bin_preds, bin_targets, type_preds, type_targets)

print(f'Test Loss: {test_loss:.4f}')
print_report(bin_preds, bin_targets, type_preds, type_targets)

In [ ]:
metrics_df = pd.DataFrame({
    'Metric': list(metrics.keys()),
    'Value':  [round(v, 4) for v in metrics.values()]
})
metrics_df

## 4. Visualizations

In [ ]:
loss_curve_path = os.path.join('results', 'loss_curves.png')
if os.path.exists(loss_curve_path):
    img = mpimg.imread(loss_curve_path)
    plt.figure(figsize=(14, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Loss curves not found — run train.py first.')

In [ ]:
plot_confusion_matrix(type_preds, type_targets)

## 5. Custom Single-Sample Inference

Run the model on a hand-crafted machine reading. Edit the values in Example 3 to try your own.

In [ ]:
FAILURE_TYPE_NAMES = ['No Failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
TYPE_MAP = {'L': 0, 'M': 1, 'H': 2}

def predict_single(air_temp, process_temp, rot_speed, torque, tool_wear, machine_type='M'):
    continuous = np.array([[air_temp, process_temp, rot_speed, torque, tool_wear]], dtype=np.float32)
    continuous_scaled = scaler.transform(continuous)
    type_enc = np.array([[TYPE_MAP[machine_type]]], dtype=np.float32)
    features = np.concatenate([continuous_scaled, type_enc], axis=1)

    x = torch.tensor(features, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        bin_logit, type_logits = model(x)

    failure_prob  = torch.sigmoid(bin_logit).item()
    type_probs    = torch.softmax(type_logits, dim=1).squeeze().cpu().numpy()
    type_pred_idx = int(type_probs.argmax())

    return {
        'failure_probability': round(failure_prob, 4),
        'binary_prediction':   'FAILURE' if failure_prob >= 0.5 else 'No Failure',
        'predicted_type':      FAILURE_TYPE_NAMES[type_pred_idx],
        'type_probabilities':  {name: round(float(p), 4)
                                for name, p in zip(FAILURE_TYPE_NAMES, type_probs)},
    }

In [ ]:
# Example 1: Normal operating conditions
result = predict_single(
    air_temp=298.1, process_temp=308.6,
    rot_speed=1551, torque=42.8, tool_wear=0,
    machine_type='M'
)
print('--- Normal operating conditions ---')
for k, v in result.items():
    print(f'  {k}: {v}')

In [ ]:
# Example 2: High torque + high tool wear — likely failure
result = predict_single(
    air_temp=302.5, process_temp=313.0,
    rot_speed=1200, torque=68.0, tool_wear=220,
    machine_type='L'
)
print('--- High stress conditions ---')
for k, v in result.items():
    print(f'  {k}: {v}')

In [ ]:
# Example 3: Edit these values and re-run!
result = predict_single(
    air_temp=300.0,
    process_temp=310.0,
    rot_speed=1400,
    torque=50.0,
    tool_wear=100,
    machine_type='H'
)
print('--- Custom input ---')
for k, v in result.items():
    print(f'  {k}: {v}')